# Reshaping: wide to long and back

Python stage, lesson P19. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** turn columns into rows with `melt`; group by two keys; turn rows into columns with `pivot`; build the Excel PivotTable in code with `pivot_table`, and check that the total survives the trip.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each order line, with its amount, store, month, category and payment method, as built in lessons P13, P15 and P17.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])
sales = lines.merge(orders[["order_id", "store_id", "ordered_at", "payment_method"]], on="order_id", validate="many_to_one")
sales = sales.merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
sales = sales.assign(month=pd.to_datetime(sales["ordered_at"]).dt.month)

## Unit 1: melt: columns into rows

**Predict first.** how many rows will `long` have, and what will its three columns be called?

In [ ]:
wide_orders = pd.DataFrame({
    "store_id": [1, 2, 3],
    "card": [421, 400, 180],
    "cash": [91, 107, 55],
    "mobile": [217, 209, 120],
})
long = wide_orders.melt(id_vars="store_id")
print(long.shape)
print(long.columns.tolist())

**Predict first.** what will the three totals be, and what will their sum be?

In [ ]:
long = wide_orders.melt(id_vars="store_id", var_name="payment_method", value_name="orders")
print(long.groupby("payment_method")["orders"].sum())
print(long["orders"].sum())

## Unit 2: Group by two keys

**Predict first.** there are 3 stores and 6 months. How many rows will `store_month` have?

In [ ]:
store_month = sales.groupby(["store_id", "month"])["amount"].sum().reset_index()
print(store_month.shape)
print(store_month.head(4))

## Unit 3: pivot: rows into columns

**Predict first.** what shape will `wide` have, and what will the row of Harbor, store 3, hold in its first three months?

In [ ]:
wide = store_month.pivot(index="store_id", columns="month", values="amount")
print(wide.shape)
print(wide.loc[3].tolist())

**Predict first.** `sales` has many rows for the same store and month. What will `pivot` do with them: add them up, keep the first, or stop?

In [ ]:
show_error('sales.pivot(index="store_id", columns="month", values="amount")', short=True)

## Unit 4: pivot_table: the PivotTable in code

**Predict first.** the table has 3 stores and 4 categories. What shape will it really have, and what will the cell in the bottom right corner hold?

In [ ]:
by_category = sales.pivot_table(index="store_id", columns="category", values="amount", aggfunc="sum", margins=True)
print(by_category.shape)
print(by_category.round(2))

**Predict first.** for store 1, the coffee cell of the table above was 4,507.75. What kind of number will the coffee cell hold now?

In [ ]:
by_default = sales.pivot_table(index="store_id", columns="category", values="amount")
print(by_default.round(2).loc[1])

## Unit 5: Check: there and back

**Predict first.** how many rows will `back` have, how many of them hold a number, and what will the numbers add up to?

In [ ]:
back = wide.reset_index().melt(id_vars="store_id", var_name="month", value_name="amount")
print(len(back), back["amount"].notna().sum(), back["amount"].sum())

## Finish it

### Exercise 1: Finish it

**Task:** What share of the orders of the customers who live in Alameda were taken at Harbor, store 3? `counts` holds the orders of each home city and store, one row for each pair; the walk-in order has no city, so it is in no group. Replace the first `None` with `counts` as a wide table, one row for each city and one column for each store, then set `answer_1` to the share, a number between 0 and 1.

**Expected:** one decimal number between 0 and 1.

In [ ]:
home = orders.merge(customers[["customer_id", "city"]], on="customer_id", how="left", validate="many_to_one")
counts = home.groupby(["city", "store_id"]).size().reset_index(name="orders")  # given: one row for each city and store
wide_home = None  # replace None: counts as a wide table, one row for each city
answer_1 = None  # then set answer_1 to the Alameda orders taken at store 3, divided by all Alameda orders

In [ ]:
_ = check("Exercise 1", "answer_1", 314 / 404, traps={314 / 355: "You divided by the column of store 3, all the orders taken there. The share of the Alameda customers' orders is a share of the row of Alameda."})

<details>
<summary>Hint</summary>

A pair of city and store is one row of `counts`, and `pivot` makes the stores the columns. Which total do you want, the one down a column or the one along a row?

</details>

<details>
<summary>Solution</summary>

```python
home = orders.merge(customers[["customer_id", "city"]], on="customer_id", how="left", validate="many_to_one")
counts = home.groupby(["city", "store_id"]).size().reset_index(name="orders")
wide_home = counts.pivot(index="city", columns="store_id", values="orders")
answer_1 = wide_home.loc["Alameda", 3] / wide_home.loc["Alameda"].sum()
```

The answer is 0.7772: 314 of the 404 orders of Alameda's customers were taken at Harbor. The row of Alameda holds 47, 43 and 314. Dividing by the column of store 3 gives 0.8845, the share of Harbor's orders that came from Alameda's customers, another question; the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** What share of the coffee sales were paid by mobile? Before you start, say in one sentence what the table should look like: which fields are rows, which are columns and which is the number. Build it with `pivot_table` on `sales`, and set `answer_2` to the share, a number between 0 and 1. Write the code in the empty cell below and set `answer_2`.

**Expected:** one decimal number between 0 and 1.

In [ ]:
_ = check("Exercise 2", "answer_2", 3053.0 / 10807.0, traps={0.3241753347172825: "The cells hold averages of a line, not totals. Which argument of pivot_table says how the rows of a cell are added up?"})

<details>
<summary>Hint</summary>

The rows of the table are the categories, the columns are the payment methods. Unit 4 shows what the table holds when you do not say how to add up.

</details>

<details>
<summary>Solution</summary>

```python
pay = sales.pivot_table(index="category", columns="payment_method", values="amount", aggfunc="sum")
answer_2 = pay.loc["coffee", "mobile"] / pay.loc["coffee"].sum()
```

The answer is 0.2825: the cells of the coffee row are 6,298.00 card, 1,456.00 cash and 3,053.00 mobile, 10,807.00 in all. Without `aggfunc` the cells hold the average of a line, about 6.58, 6.68 and 6.36, and the share comes out as 0.3242; the check cell says so.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants the average monthly sales of Harbor, store 3, over the months in which it traded. They built the wide table with an empty cell filled in, and their average is 767.71. Fix the line, then set `answer_3` to Harbor's average, a decimal number. `store_month` is the table of unit 2.

**Expected:** one decimal number.

In [ ]:
wide_fixed = store_month.pivot_table(index="store_id", columns="month", values="amount", fill_value=0)  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 = wide_fixed.loc[3].mean()

In [ ]:
_ = check("Exercise 3", "answer_3", 4606.25 / 3, traps={4606.25 / 6: "Harbor did not sell 0 dollars in January to March, it was not open. A 0 is counted in the average, an empty cell is skipped. Which argument of pivot_table wrote the zeros?"})

<details>
<summary>Hint</summary>

Print `wide_fixed.loc[3]`. What do the first three months hold, and would the average skip them if they were empty? The pandas page for `pivot_table` describes `fill_value`.

</details>

<details>
<summary>Solution</summary>

```python
wide_fixed = store_month.pivot_table(index="store_id", columns="month", values="amount")
answer_3 = wide_fixed.loc[3].mean()
```

The answer is 1,535.42: Harbor's 4,606.25 over the 3 months it traded. `fill_value=0` wrote a zero where there was no row, so the average ran over 6 months and gave 767.71, half of the true figure; the check cell says so.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")